# Domain-Adversarial NN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import lightning as L
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 1. setup and data loading

In [ ]:
print("="*60)
print("DOMAIN-ADVERSARIAL NEURAL NETWORK (DANN) - IMPROVED")
print("CROSS-DATASET AUTISM CLASSIFICATION")
print("="*60)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Load balanced datasets
print("\nLoading balanced datasets...")
c4_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced.csv')
ybt_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_balanced_standardized.csv')

print(f"C4 balanced shape: {c4_balanced.shape}")
print(f"YBT balanced shape: {ybt_balanced.shape}")

# 2. data prep

In [ ]:
print("\n" + "="*60)
print("ENHANCED DATA PREPARATION AND ANALYSIS")
print("="*60)

# Identify common features (excluding target)
exclude_cols = ['autism_target']
c4_features = [col for col in c4_balanced.columns if col not in exclude_cols]
ybt_features = [col for col in ybt_balanced.columns if col not in exclude_cols]

# Find common features
common_features = list(set(c4_features) & set(ybt_features))
print(f"Common features: {len(common_features)}")

# Show some common feature names
print(f"Sample common features: {common_features[:10]}")

# Prepare data
X_c4 = c4_balanced[common_features].values
y_c4 = c4_balanced['autism_target'].values
X_ybt = ybt_balanced[common_features].values
y_ybt = ybt_balanced['autism_target'].values

print(f"C4 features shape: {X_c4.shape}")
print(f"YBT features shape: {X_ybt.shape}")

# Check class distributions
print(f"\nClass distribution in C4:")
print(f"Class 0: {np.sum(y_c4 == 0)} ({np.mean(y_c4 == 0)*100:.1f}%)")
print(f"Class 1: {np.sum(y_c4 == 1)} ({np.mean(y_c4 == 1)*100:.1f}%)")

print(f"\nClass distribution in YBT:")
print(f"Class 0: {np.sum(y_ybt == 0)} ({np.mean(y_ybt == 0)*100:.1f}%)")
print(f"Class 1: {np.sum(y_ybt == 1)} ({np.mean(y_ybt == 1)*100:.1f}%)")

# Check for data quality issues
print(f"\nData quality checks:")
print(f"C4 data range: [{X_c4.min():.3f}, {X_c4.max():.3f}]")
print(f"YBT data range: [{X_ybt.min():.3f}, {X_ybt.max():.3f}]")
print(f"C4 data mean: {X_c4.mean():.3f}, std: {X_c4.std():.3f}")
print(f"YBT data mean: {X_ybt.mean():.3f}, std: {X_ybt.std():.3f}")

# Check for NaN values
print(f"\nNaN values:")
print(f"C4 NaN count: {np.isnan(X_c4).sum()}")
print(f"YBT NaN count: {np.isnan(X_ybt).sum()}")

# Check for infinite values
print(f"\nInfinite values:")
print(f"C4 inf count: {np.isinf(X_c4).sum()}")
print(f"YBT inf count: {np.isinf(X_ybt).sum()}")

# Data standardization with robust scaling
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()  # More robust to outliers
X_c4_scaled = scaler.fit_transform(X_c4)
X_ybt_scaled = scaler.transform(X_ybt)

print(f"\nAfter robust standardization:")
print(f"C4 scaled mean: {X_c4_scaled.mean():.3f}, std: {X_c4_scaled.std():.3f}")
print(f"YBT scaled mean: {X_ybt_scaled.mean():.3f}, std: {X_ybt_scaled.std():.3f}")

# Check for extreme values after scaling
print(f"\nExtreme values after scaling:")
print(f"C4 scaled range: [{X_c4_scaled.min():.3f}, {X_c4_scaled.max():.3f}]")
print(f"YBT scaled range: [{X_ybt_scaled.min():.3f}, {X_ybt_scaled.max():.3f}]")

# Clip extreme values
X_c4_scaled = np.clip(X_c4_scaled, -10, 10)
X_ybt_scaled = np.clip(X_ybt_scaled, -10, 10)

print(f"\nAfter clipping extreme values:")
print(f"C4 clipped range: [{X_c4_scaled.min():.3f}, {X_c4_scaled.max():.3f}]")
print(f"YBT clipped range: [{X_ybt_scaled.min():.3f}, {X_ybt_scaled.max():.3f}]")

# Split C4 data for training/validation
X_train, X_val, y_train, y_val = train_test_split(
    X_c4_scaled, y_c4, test_size=0.2, stratify=y_c4, random_state=42
)

print(f"\nTraining set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"YBT test set: {X_ybt_scaled.shape}")

# Check training set class distribution
print(f"\nTraining set class distribution:")
print(f"Class 0: {np.sum(y_train == 0)} ({np.mean(y_train == 0)*100:.1f}%)")
print(f"Class 1: {np.sum(y_train == 1)} ({np.mean(y_train == 1)*100:.1f}%)")

# 2.5 (feature analysis and expansion to ensure datasets match)

In [ ]:
print("\n" + "="*60)
print("FEATURE ANALYSIS AND EXPANSION")
print("="*60)

# Analyze feature overlap
print(f"\nFeature Analysis:")
print(f"C4 total features: {len(c4_features)}")
print(f"YBT total features: {len(ybt_features)}")
print(f"Common features: {len(common_features)}")

# Show feature categories
c4_eq_features = [f for f in c4_features if 'eq' in f.lower()]
c4_aq_features = [f for f in c4_features if 'aq' in f.lower()]
c4_sqr_features = [f for f in c4_features if 'sqr' in f.lower()]
c4_spq_features = [f for f in c4_features if 'spq' in f.lower()]

ybt_eq_features = [f for f in ybt_features if 'eq' in f.lower()]
ybt_aq_features = [f for f in ybt_features if 'aq' in f.lower()]
ybt_sqr_features = [f for f in ybt_features if 'sqr' in f.lower()]

print(f"\nFeature Categories:")
print(f"C4 - EQ features: {len(c4_eq_features)}")
print(f"C4 - AQ features: {len(c4_aq_features)}")
print(f"C4 - SQR features: {len(c4_sqr_features)}")
print(f"C4 - SPQ features: {len(c4_spq_features)}")

print(f"YBT - EQ features: {len(ybt_eq_features)}")
print(f"YBT - AQ features: {len(ybt_aq_features)}")
print(f"YBT - SQR features: {len(ybt_sqr_features)}")

# Create additional aggregate features
print(f"\nCreating additional aggregate features...")

# Create missing aggregate features for YBT
if 'eq_total' in c4_features and 'eq_total' not in ybt_balanced.columns:
    print("  - Creating eq_total for YBT")
    eq_cols = [f for f in ybt_features if 'eq' in f.lower()]
    if eq_cols:
        ybt_balanced['eq_total'] = ybt_balanced[eq_cols].sum(axis=1)

if 'aq_total' in c4_features and 'aq_total' not in ybt_balanced.columns:
    print("  - Creating aq_total for YBT")
    aq_cols = [f for f in ybt_features if 'aq' in f.lower()]
    if aq_cols:
        ybt_balanced['aq_total'] = ybt_balanced[aq_cols].sum(axis=1)

if 'sqr_total' in c4_features and 'sqr_total' not in ybt_balanced.columns:
    print("  - Creating sqr_total for YBT")
    sqr_cols = [f for f in ybt_features if 'sqr' in f.lower()]
    if sqr_cols:
        ybt_balanced['sqr_total'] = ybt_balanced[sqr_cols].sum(axis=1)

# Create interaction features
if 'age_x_aq' in c4_features and 'age_x_aq' not in ybt_balanced.columns:
    print("  - Creating age_x_aq for YBT")
    if 'age' in ybt_balanced.columns and 'aq_total' in ybt_balanced.columns:
        ybt_balanced['age_x_aq'] = ybt_balanced['age'] * ybt_balanced['aq_total']

if 'age_x_eq' in c4_features and 'age_x_eq' not in ybt_balanced.columns:
    print("  - Creating age_x_eq for YBT")
    if 'age' in ybt_balanced.columns and 'eq_total' in ybt_balanced.columns:
        ybt_balanced['age_x_eq'] = ybt_balanced['age'] * ybt_balanced['eq_total']

# Re-check common features after additions
ybt_features_updated = [col for col in ybt_balanced.columns if col not in exclude_cols]
common_features_updated = list(set(c4_features) & set(ybt_features_updated))

print(f"\nUpdated common features: {len(common_features_updated)}")
if len(common_features_updated) > len(common_features):
    print(f"✓ Successfully added {len(common_features_updated) - len(common_features)} features!")
    common_features = common_features_updated
    # Update data preparation
    X_c4 = c4_balanced[common_features].values
    X_ybt = ybt_balanced[common_features].values
    # Re-standardize
    X_c4_scaled = scaler.fit_transform(X_c4)
    X_ybt_scaled = scaler.transform(X_ybt)
    # Re-split
    X_train, X_val, y_train, y_val = train_test_split(
        X_c4_scaled, y_c4, test_size=0.2, stratify=y_c4, random_state=42
    )
else:
    print("  - No additional features could be created automatically")

print(f"\nFinal feature count: {len(common_features)}")
print(f"Final data shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_ybt_scaled: {X_ybt_scaled.shape}")

# 3. gradient reversal layer

In [ ]:
print("\n" + "="*60)
print("IMPROVED GRADIENT REVERSAL LAYER")
print("="*60)

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

# 4. DA classifier 

In [ ]:
print("\n" + "="*60)
print("IMPROVED DOMAIN-ADVERSARIAL CLASSIFIER")
print("="*60)

class ImprovedDomainAdversarialClassifier(L.LightningModule):
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], dropout_rate=0.2, 
                 learning_rate=0.0001, alpha=1.0, domain_weight=0.1):
        super().__init__()
        self.save_hyperparameters()
        
        # Feature extractor (shared between tasks)
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        
        self.feature_extractor = nn.Sequential(*layers)
        
        # Classifier (for autism prediction)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dims[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Domain discriminator (for domain prediction)
        self.domain_discriminator = nn.Sequential(
            GradientReversalLayer(alpha),
            nn.Linear(hidden_dims[-1], 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.domain_weight = domain_weight
        
        # Loss functions
        self.class_criterion = nn.BCELoss()
        self.domain_criterion = nn.BCELoss()
        
    def forward(self, x):
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        domain_output = self.domain_discriminator(features)
        return class_output, domain_output
    
    def training_step(self, batch, batch_idx):
        x, y, domain = batch
        
        # Forward pass
        class_output, domain_output = self(x)
        
        # Classification loss
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        
        # Domain discrimination loss
        domain_loss = self.domain_criterion(domain_output.squeeze(), domain.float())
        
        # Total loss (classification + domain adversarial)
        total_loss = class_loss + self.domain_weight * domain_loss
        
        # Log metrics
        self.log('train_class_loss', class_loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_domain_loss', domain_loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_total_loss', total_loss, on_step=True, on_epoch=True, prog_bar=True)
        
        # Calculate classification metrics
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        self.log('train_f1', f1, on_epoch=True, prog_bar=True)
        
        return total_loss
    
    def validation_step(self, batch, batch_idx):
        x, y, domain = batch
        
        # Forward pass
        class_output, domain_output = self(x)
        
        # Classification loss
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        
        # Domain discrimination loss
        domain_loss = self.domain_criterion(domain_output.squeeze(), domain.float())
        
        # Total loss
        total_loss = class_loss + self.domain_weight * domain_loss
        
        # Log metrics
        self.log('val_class_loss', class_loss, on_epoch=True, prog_bar=True)
        self.log('val_domain_loss', domain_loss, on_epoch=True, prog_bar=True)
        self.log('val_total_loss', total_loss, on_epoch=True, prog_bar=True)
        
        # Calculate classification metrics
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        
        # Handle AUC calculation safely
        try:
            auc = roc_auc_score(y.cpu(), class_output.squeeze().detach().cpu())
        except ValueError:
            auc = 0.5  # Default to random if only one class
        
        self.log('val_f1', f1, on_epoch=True, prog_bar=True)
        self.log('val_auc', auc, on_epoch=True, prog_bar=True)
        
        return total_loss
    
    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.7, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_total_loss",
            },
        }

 # domain aware data module

In [ ]:
print("\n" + "="*60)
print("IMPROVED DOMAIN-AWARE DATA MODULE")
print("="*60)

class ImprovedDomainAdaptationDataModule(L.LightningDataModule):
    def __init__(self, X_train, X_val, y_train, y_val, X_target, y_target, batch_size=64):
        super().__init__()
        self.X_train = X_train
        self.X_val = X_val
        self.y_train = y_train
        self.y_val = y_val
        self.X_target = X_target  # YBT data
        self.y_target = y_target
        self.batch_size = batch_size
    
    def setup(self, stage=None):
        # Source domain (C4) - domain label 0
        self.train_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_train),
            torch.LongTensor(self.y_train),
            torch.zeros(len(self.X_train))  # Domain label 0 for source
        )
        
        self.val_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_val),
            torch.LongTensor(self.y_val),
            torch.zeros(len(self.X_val))  # Domain label 0 for source
        )
        
        # Target domain (YBT) - domain label 1
        self.target_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_target),
            torch.LongTensor(self.y_target),
            torch.ones(len(self.X_target))  # Domain label 1 for target
        )
    
    def train_dataloader(self):
        # Combine source and target data for training
        combined_dataset = torch.utils.data.ConcatDataset([
            self.train_dataset, self.target_dataset
        ])
        return torch.utils.data.DataLoader(
            combined_dataset, 
            batch_size=self.batch_size, 
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
    
    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size, 
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

# 6. model training

In [ ]:
print("\n" + "="*60)
print("IMPROVED DANN MODEL TRAINING")
print("="*60)

# Initialize model with better architecture for small feature set
input_dim = len(common_features)
model = ImprovedDomainAdversarialClassifier(
    input_dim=input_dim,
    hidden_dims=[128, 64, 32],  # Smaller network for limited features
    dropout_rate=0.3,
    learning_rate=0.0005,  # Slightly higher learning rate
    alpha=1.0,
    domain_weight=0.05  # Reduce domain weight to focus on classification
)

# Initialize data module
data_module = ImprovedDomainAdaptationDataModule(
    X_train, X_val, y_train, y_val, X_ybt_scaled, y_ybt, batch_size=64
)

# Initialize trainer with better configuration
trainer = L.Trainer(
    max_epochs=100,  # More epochs for small dataset
    accelerator='auto',
    devices=1,
    callbacks=[
        L.pytorch.callbacks.EarlyStopping(
            monitor='val_f1',
            patience=15,  # More patience
            mode='max',
            verbose=True
        ),
        L.pytorch.callbacks.ModelCheckpoint(
            monitor='val_f1',
            mode='max',
            save_top_k=3,
            filename='best_dann_improved_{epoch:02d}_{val_f1:.3f}',
            verbose=True
        ),
        L.pytorch.callbacks.LearningRateMonitor(logging_interval='epoch')
    ],
    log_every_n_steps=25,
    enable_progress_bar=True,
    enable_model_summary=True,
    deterministic=True
)

# Train model
print("Starting improved DANN training...")
trainer.fit(model, data_module)

print("Improved DANN training completed!")

# 7. model eval

In [ ]:
print("\n" + "="*60)
print("ENHANCED MODEL EVALUATION")
print("="*60)

# Load best model
best_model_path = trainer.checkpoint_callback.best_model_path
print(f"Loading best model from: {best_model_path}")

# Load model
model = ImprovedDomainAdversarialClassifier.load_from_checkpoint(best_model_path)
model.eval()

# Evaluate on validation set
val_predictions = []
val_probs = []
val_targets = []

with torch.no_grad():
    for batch in data_module.val_dataloader():
        x, y, domain = batch
        device = next(model.parameters()).device
        x = x.to(device)
        class_output, domain_output = model(x)
        val_probs.extend(class_output.squeeze().cpu().numpy())
        val_predictions.extend((class_output.squeeze() > 0.5).cpu().numpy())
        val_targets.extend(y.cpu().numpy())

val_probs = np.array(val_probs)
val_predictions = np.array(val_predictions)
val_targets = np.array(val_targets)

print("\nValidation Set Performance:")
print(classification_report(val_targets, val_predictions, zero_division=0))
print(f"ROC-AUC: {roc_auc_score(val_targets, val_probs):.3f}")

# Plot validation predictions distribution
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(val_probs, bins=50, alpha=0.7)
plt.title('Validation Prediction Probabilities Distribution')
plt.xlabel('Prediction Probability')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.hist(val_probs[val_targets == 0], bins=30, alpha=0.7, label='Class 0', density=True)
plt.hist(val_probs[val_targets == 1], bins=30, alpha=0.7, label='Class 1', density=True)
plt.title('Prediction Probabilities by Class')
plt.xlabel('Prediction Probability')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

# 8. cross dataset testing

In [ ]:
print("\n" + "="*60)
print("CROSS-DATASET TESTING (C4 → YBT) - IMPROVED")
print("="*60)

# Prepare YBT test data
X_ybt_tensor = torch.FloatTensor(X_ybt_scaled)
ybt_dataset = torch.utils.data.TensorDataset(
    X_ybt_tensor, torch.LongTensor(y_ybt), torch.ones(len(X_ybt_scaled))
)
ybt_dataloader = torch.utils.data.DataLoader(ybt_dataset, batch_size=64, shuffle=False)

# Evaluate on YBT
ybt_predictions = []
ybt_probs = []
ybt_targets = []

model.eval()
device = next(model.parameters()).device

with torch.no_grad():
    for batch in ybt_dataloader:
        x, y, domain = batch
        x = x.to(device)
        class_output, domain_output = model(x)
        ybt_probs.extend(class_output.squeeze().cpu().numpy())
        ybt_predictions.extend((class_output.squeeze() > 0.5).cpu().numpy())
        ybt_targets.extend(y.cpu().numpy())

ybt_probs = np.array(ybt_probs)
ybt_predictions = np.array(ybt_predictions)
ybt_targets = np.array(ybt_targets)

print("\nYBT Test Set Performance:")
print(classification_report(ybt_targets, ybt_predictions, zero_division=0))
print(f"ROC-AUC: {roc_auc_score(ybt_targets, ybt_probs):.3f}")

# Threshold optimization for YBT
from sklearn.metrics import precision_recall_curve
prec, rec, thresholds = precision_recall_curve(ybt_targets, ybt_probs)
f1s = 2 * (prec * rec) / (prec + rec + 1e-8)
best_thresh_idx = np.argmax(f1s)
best_threshold = thresholds[best_thresh_idx]

print(f"\nBest threshold for YBT: {best_threshold:.3f}")
ybt_predictions_optimal = (ybt_probs >= best_threshold).astype(int)
print(f"F1 at optimal threshold: {f1_score(ybt_targets, ybt_predictions_optimal):.3f}")

# Plot YBT predictions distribution
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(ybt_probs, bins=50, alpha=0.7)
plt.title('YBT Prediction Probabilities Distribution')
plt.xlabel('Prediction Probability')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.hist(ybt_probs[ybt_targets == 0], bins=30, alpha=0.7, label='Class 0', density=True)
plt.hist(ybt_probs[ybt_targets == 1], bins=30, alpha=0.7, label='Class 1', density=True)
plt.title('YBT Prediction Probabilities by Class')
plt.xlabel('Prediction Probability')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

# 9. comparison with prev models

In [ ]:
print("\n" + "="*60)
print("COMPREHENSIVE COMPARISON AND ANALYSIS")
print("="*60)

results_comparison = {
    'Model': ['Random Forest', 'Neural Network', 'TabNet', 'DANN (Original)', 'DANN (Improved)'],
    'F1_Score': [0.619, 0.667, 0.667, 0.667, f1_score(ybt_targets, ybt_predictions_optimal)],
    'ROC_AUC': [0.325, 0.523, 0.388, 0.500, roc_auc_score(ybt_targets, ybt_probs)],
    'Threshold': [0.159, 0.157, 0.116, 0.505, best_threshold]
}

comparison_df = pd.DataFrame(results_comparison)
print("\nPerformance Comparison:")
print(comparison_df)

# Save model
os.makedirs('/Users/eb2007/playground/bullpy/c4_play2/models', exist_ok=True)
torch.save(model.state_dict(), '/Users/eb2007/playground/bullpy/c4_play2/models/dann_improved.pth')

print("\nImproved DANN model saved successfully!")
print("Domain adaptation experiment completed!")

# Additional analysis
print(f"\nDetailed Analysis:")
print(f"- Model stopped at epoch: {trainer.current_epoch}")
print(f"- Best validation F1: {trainer.checkpoint_callback.best_model_score:.3f}")
print(f"- Final learning rate: {trainer.optimizers[0].param_groups[0]['lr']:.6f}")
print(f"- Prediction range: [{ybt_probs.min():.3f}, {ybt_probs.max():.3f}]")
print(f"- Mean prediction: {ybt_probs.mean():.3f}")
print(f"- Std prediction: {ybt_probs.std():.3f}")